# 研究架構

::: {.callout-tip}

投影片操作：**Alt + 點擊** 可縮放任何圖片/表格（reveal.js zoom）；`O` 鍵總覽、`F` 全螢幕。

:::

本研究檢驗配對交易兩個環節能否以機器學習與深度學習改良：

| | 檢定的對象 | 對照設計 | 結果 |
| :--- | :--- | :--- | :--- |
| **組合系統** | 動態分群 + **門檻選擇** vs GICS + **固定門檻** | 3 組（排序已對齊）× 兩個期間 | **5/6 顯著**（BH 校正後） |
| **命題 1** | **機器學習分群**能建立優於傳統產業分類的配對搜尋空間 | 4 分組 × 3 排序矩陣（分組為唯一變因） | **未獲支持** |
| **命題 2** | 以**學習法選擇門檻**優於固定門檻規則 | 5 種配對底 × 交易端（交易端為唯一變因） | **獲得支持** |

> 三者不衝突：組合系統跨過門檻，是兩個同向效果**相加**的結果。
> **完整系統顯著優於傳統基準，但把功勞歸給哪一半，資料還答不出來。**

**共同控制條件**：混合特徵（報酬 PCA ⊕ PIT 基本面 ⊕ GICS one-hot）、
共整合篩選（ADF 0.05 + OU 半衰期 + Hurst）、S&P 500 歷史成分股 2000–2025、
形成期 252 日 / 交易期 126 日 / 滾動 21 日、交易成本單邊 0.29%。

::: {.callout-important}

### 本研究的推論口徑

跨策略比較以**逐日報酬差 + 循環 block bootstrap**（$L$=126，10,000 次重抽）為準，
抽樣單位為**時間**，一切績效主張皆以**全網格等權組合**為口徑。

早期版本以「15 個參數網格」為抽樣單位做配對 $t$ 檢定，但該 15 格共用同一份資料、
同一段期間與同一批配對（$top_n$=10 與 20 共用 10 個配對；三種停損是同一批交易的
不同出場規則），觀測高度相關，獨立性假設不成立（pseudo-replication，
有效樣本數 $\approx$ 1 條回測路徑）。該版本結果已降為描述性附錄。

**改用時間維度後，效果量幾乎不變**（命題 2 年化 +0.787pp vs 舊 +0.732pp），
但**推論結論改變**：命題 1 原稱「5 組顯著較差」者，經校正後無一顯著。

:::

# 方法論：形成期四層架構

策略 = 四個可獨立替換的層之組態，由中性組裝器依參數組裝：

| 層 | 職責 | 本研究採用的選項 |
| :--- | :--- | :--- |
| **特徵** | 個股 → 特徵向量 | 報酬 PCA 因子載荷 ⊕ 基本面 ⊕ 產業 one-hot |
| **分組** | 特徵 → 配對搜尋空間 | GICS 產業／HDBSCAN／Agglomerative／K-means |
| **排序** | 群內配對 → 優先序 | SSD／DTW／SSD-DTW-PCA |
| **篩選** | 統計檢定淘汰 | ADF 共整合 + OU 半衰期 + Hurst |

交易期則為 **Z-Score 狀態機**（規則型基準）或 **DL-THR 門檻選擇式**（Kim & Kim 2019 風格）。

> 此架構使「分組」「排序」「交易端」皆可作為單變因替換，是兩大命題能乾淨對照的前提。
> 實作正確性以數值回歸測試保證：組裝器復現原生策略的結果為**逐位元相同**。


# 參考文獻與方法對應

## 分群方法
> Campello, Moulavi & Sander (2013). Density-based clustering based on hierarchical density estimates. *PAKDD*.
> Ward (1963). Hierarchical grouping to optimize an objective function. *JASA*, **58**(301).
> MacQueen (1967). Some methods for classification and analysis of multivariate observations.

HDBSCAN（自動群數、噪音標記）、Agglomerative（dendrogram 分位數校準）、
K-means（群數對齊同期 Agglomerative，使量級可比）。

## 排序準則
> Gatev, Goetzmann & Rouwenhorst (2006). Pairs trading. *RFS*, **19**(3).　📄 `ref/2006-...pdf`
> 許鈞翔 (2025)。最小距離法結合動態時間校正之配對交易研究。　📄 `ref/2025-...pdf`

SSD（同步距離）、DTW（Sakoe-Chiba 時間扭曲）、SSD-DTW-PCA（兩距離 PCA 融合）。

## 特徵與交易端
> Avellaneda & Lee (2010). Statistical arbitrage in the U.S. equities market. *Quantitative Finance*, **10**(7).　📄
> Hong & Hwang (2021). In search of pairs using firm fundamentals. *EJF*, **29**(5).　📄
> Kim & Kim (2019). Optimizing the pairs-trading strategy using DRL with trading and stop-loss boundaries. *Complexity*.　📄

報酬 PCA 因子載荷（Avellaneda & Lee）、基本面配對（Hong & Hwang）、
DL-THR 門檻選擇式交易（Kim & Kim）。


# 命題 1：機器學習分群 vs 傳統產業分類

**設計**：固定特徵、篩選、Z-Score 交易端；變動分組方法 × 排序準則。

## 各格網格最佳值（最佳年化 / 最佳 Sharpe）

| 分組＼排序 | SSD | DTW | SSD-DTW-PCA |
| :--- | :---: | :---: | :---: |
| **GICS（傳統）** | 1.66% / 0.20 | 1.10% / 0.18 | 1.65% / **0.35** |
| **HDBSCAN** | 1.29% / 0.17 | 0.89% / 0.15 | **1.70%** / 0.23 |
| **Agglomerative** | **1.77% / 0.26** | 0.60% / 0.11 | 1.13% / 0.18 |
| **K-means** | 1.31% / 0.20 | −0.06% / 0.03 | 0.67% / 0.12 |

若僅看此表，會得到「Agglomerative × SSD 優於 GICS × SSD（1.77% vs 1.66%）」的印象。
然而兩者皆為 15 格中的最大值，該差距未經檢定。


## 命題 1 的假設檢定：無顯著差異，且無法主張相當

逐日報酬差 $\Delta r_t = r_{ML,t} - r_{GICS,t}$（15 格等權組合，6,287 交易日），
循環 block bootstrap（$L$=126），$H_0$：兩者績效相同。9 組比較以 Benjamini-Hochberg 控制 FDR。

| 分組＼排序 | SSD | DTW | SSD-DTW-PCA |
| :--- | :---: | :---: | :---: |
| HDBSCAN | **+0.368** (p=0.261) | **+0.246** (p=0.468) | **+0.160** (p=0.598) |
| Agglomerative | **+0.369** (p=0.363) | −0.452 (p=0.305) | −0.556 (p=0.143) |
| K-means | **+0.025** (p=0.954) | −0.647 (p=0.152) | −0.816 (p=0.063) |

*年化報酬差（百分點，ML − GICS）；原始 $p$ 值。**BH 校正後最小 $p$ = 0.455***

**9 組比較中，校正後 0 組顯著**；方向上 **ML 優 5 組、GICS 優 4 組**。
（9 組彼此不獨立——每三組共用同一個 GICS 臂，故不對方向一致性施加正式檢定。）

::: {.callout-note}

### 這個 null 已排除「資料太差」的替代解釋

2026-08-10 前的結果跑在覆蓋率僅約四分之一的基本面上（市值 10.5%），
2 維基本面九成是**產業中位數插補**，等同第二份產業 one-hot。
資料回補後（39.7%）方向由 **1:8 轉為 5:4**，**但仍無一顯著**。
改善集中於 SSD 排序與 HDBSCAN；DTW／SDP 配 Agglomerative／K-means 四組幾乎未動。

:::

::: {.callout-important}

### 「不顯著」不等於「兩者相當」——看區間寬度

**9 組的 95% 信賴區間全部涵蓋 0，寬度介於 1.2 ~ 1.9 個百分點**：

| | 最窄 | 最寬 |
| :--- | :---: | :---: |
| 組別 | HDBSCAN × SDP | K-means × SSD |
| 95% CI (pp) | [−0.41, +0.77] | [−0.97, +0.94] |

而 GICS 參照臂自身的等權組合年化報酬為 **−0.33% / −0.40% / −0.11%**（SSD / DTW / SDP）
→ **區間寬度是參照臂整體績效的數倍**。

亦即資料同時無法排除「ML 優於 GICS 達 0.9 pp」與「劣於 GICS 達 1.0 pp」。

**結論：本研究對此比較檢定力不足，既無法證明 ML 較優，亦無法主張兩者相當。**

:::

> **命題 1 未獲支持**：主張「ML 分群能建立更優的配對搜尋空間」，未能拒絕虛無假設。
> 此為「未獲支持」，**非**「GICS 顯著較優」。

## 命題 1 為何失敗：兩項結構性代價

ML 分群與 GICS 分組的差異可完全歸結為兩項可測量的結構變化：

| 分組法 | 跨產業配對比例 | 期均配對數 |
| :--- | :---: | :---: |
| **GICS（傳統）** | **0.0%**（定義上不可能） | 15.0 |
| HDBSCAN | **24.9%** | 15.5 |
| Agglomerative | 10.7% | 11.5 |
| K-means | 12.3% | **8.7** |

**代價一：破壞產業一致性。** ML 分群依特徵空間的幾何鄰近性分組，
會將不同產業但特徵相近的股票歸為同群，產生 11–25% 的跨產業配對。
產業一致性是配對交易均值回歸假設的經濟基礎（共同的需求衝擊、成本結構、監管環境）——
跨產業配對的價差即使歷史上接近，也缺乏使其回歸的經濟機制。
惟須註明：2026-08-10 基本面資料回補後，**9 組對照經 BH 校正後全部不顯著**，
故本節的兩項機制是**形成期結構**的可測量差異，**不是**績效差異的已證實成因。

**代價二：縮減候選池。** 分群將母體切成數十個小群，
群內可列舉的配對數遠少於 GICS 的 11 個大產業。
K-means 期均僅能選出 8.7 對（目標 20 對），無法填滿——
排序準則被迫接受品質更差的候選。

> 此機制與本研究另一項發現同源：收緊 ADF 門檻至 0.01 亦使績效下降
> （1.77% → 1.27%），因為更嚴的篩選同樣迫使排序往距離更遠的候選尋找。
> **兩者共同指向：候選池的充足程度是配對品質的關鍵限制因素。**


## 粒度掃描：分群演算法從來不是關鍵變因

前頁的兩項代價在原設計中互相混淆——三種分群演算法產生的群數本就不同，
無法斷定落後源於「演算法」還是「粒度」。

**受控實驗**：固定 Agglomerative 演算法、特徵、SSD 排序、篩選與交易端，
唯一變因為切割門檻分位 $q$（分位越高 → 群越少越大 → 候選池越大）。

| 門檻分位 | 期均配對數 | 跨產業% | 最佳年化 | 網格均 Sharpe | vs GICS $p$ |
| :---: | :---: | :---: | :---: | :---: | :---: |
| 50 | 6.2 | 9.7% | −0.13% | −0.237 | 0.727 |
| **60** | 8.7 | 10.3% | **1.92%** | **−0.176** | 0.102 |
| 75（基準） | 11.5 | 10.7% | 1.77% | −0.240 | 0.586 |
| 90 | **15.0** | 31.9% | 0.40% | −0.297 | 0.140 |
| 95 | 16.8 | **50.0%** | 0.38% | −0.249 | 0.772 |
| **GICS** | **15.0** | **0.0%** | 1.66% | −0.261 | — |

**發現一：粒度的影響遠大於演算法的影響。**
單一參數即使最佳年化自 −0.13% 變動至 1.92%（幅度 2.05pp）；
相較之下，固定粒度下三種分群演算法的差距僅 0.48pp（AGG 1.77 / KM 1.31 / HDB 1.29）。
**命題 1 原本比較的「分群演算法」，並非決定配對品質的主要變因。**
（五組門檻分位與 GICS 的對照 $p$ = 0.10 ~ 0.77，**無一顯著**——
本掃描刻畫的是形成期結構隨粒度的變化，不是已證實的績效差異。）

**發現二：關係為倒 U 形，非單調。** 峰值在 $q$ = 60–75，兩端皆劣化。
兩個機制反向作用——
粒度過細（$q$=50）候選池僅 6.2 對，填不滿目標而被迫接受劣質配對；
粒度過粗（$q$≥90）池雖變大，但跨產業比例自 10% 暴增至 32–50%。
候選池大小與跨產業比例的 Spearman $ho = +1.000$：**兩者在 ML 分群下無法解耦。**


## 實作原因的因子檢驗：三處差異全部消融後仍未改善

命題 1 的動機源自 Han, He & Toh (2021)（CRSP 全美股、48 動量因子 + 78 公司特徵分群、
群內做多低估/做空高估、**不施加共整合篩選**，並明文指出跨產業發散亦為利潤來源）。
本研究實作與其有三處差異，皆可能單獨壓抑該假說且從未消融：
**特徵含 12 維 GICS 產業 one-hot**、**施加共整合篩選**、**缺「不分組」零點**。

**2×2×2 因子設計**（排序固定 SSD）：

| 分組 | 篩選 | 期均/20 | 跨產業% | 全網格等權年化% |
| :--- | :---: | ---: | ---: | ---: |
| GICS 產業 | coint | 15.0 | 0.0 | −0.333 |
| AGG +one-hot | coint | 11.5 | 10.7 | −0.234 |
| AGG −one-hot | coint | 11.3 | **39.6** | −0.300 |
| 不分組 | coint | **19.4** | **75.3** | −0.245 |
| GICS 產業 | none | 20.0 | 0.0 | +0.135 |
| AGG +one-hot | none | 20.0 | 6.8 | −0.373 |
| AGG −one-hot | none | 20.0 | 25.3 | **+0.226** |
| 不分組 | none | 20.0 | 44.7 | +0.184 |

**形成期結構如預期改變**——產業先驗強度形成單調階梯（跨產業比例 0% → 10.7% → 39.6% → 75.3%），
證實 one-hot 是一個可控的「產業先驗旋鈕」；且候選池飢餓被隔離為
**「分群 × 篩選」的交互作用**——分群+篩選僅填 11.3–11.5/20，但不分組+篩選達 19.4/20。
單一因子皆不足以造成飢餓。

::: {.callout-important}

### 但績效未隨之改善

20 項單因子對照經 BH-FDR 校正後**僅 1 項顯著**（拿掉 one-hot × 無篩選，Top1 口徑
+3.07pp/年，BH $p$=0.030），且該顯著性**未能在全網格等權口徑複現**（+0.60pp，BH $p$=0.644）。
效果只出現在每期僅持一對、噪音最大的口徑——平均通常提高檢定力，此處相反，
故應視為**與假說一致但未被建立**。八種設計的等權年化全部落在 −0.37% ~ +0.23% 之間。

**本研究無法將命題 1 的否定歸因於上述實作選擇。**

:::

與文獻的差距更可能來自兩項無法以參數調整消除的根本差異：
**母體範圍**（S&P 500 約 600 檔 vs CRSP 數千檔；該文已排除「獲利來自小型股」的解釋）
與**交易機制**（群內橫斷面多空 vs 逐配對價差收斂——後者要求價差平穩性，前者不要求）。

## 交易機制的逐步歸因：方向一致，但仍不顯著

形成期實作差異被排除後，剩餘殘差之一為**交易機制**。Han et al. 押的是
**群內短期反轉**（相似兩檔上月分開、下月收斂），本研究押的是
**共整合均值回歸**（價差歷史平穩、現在偏離 2σ）——訊號、持有期、退出條件全不同。

四項差異逐步施加以維持單變因（直接完整復刻會一次改四個變因）：

| 步驟 | 該步改變 | 全網格等權年化% | Top1/SL0年化% |
| :--- | :--- | ---: | ---: |
| 起點 | SSD 距離 + OLS-β + $z$>2 / 126 日 | −0.373 | −0.909 |
| ② | β 改 1 等金額 | −0.177 | −0.370 |
| ③ | 選對準則改月報酬發散 | −0.092 | +0.207 |
| ④ | 21 日窗 + 發散即建倉 + 持有至期末 | −0.094 | **+1.369** |

::: {.callout-important}

### 每一步都改善，但沒有一步顯著

Top1 口徑單調改善（−0.909% → +1.369%，計 +2.28pp），惟逐步對照校正前最小
$p$ = 0.055、BH 校正後無一顯著；總效果等權 +0.28pp（$p$=0.923）、
Top1 +2.28pp（$p$=0.705）。`entry_z`=0 搭配持有至期末使單一配對日報酬波動極大，
檢定力極低。**完整復刻交易端後等權年化仍為 −0.094%**，離原文 24.8% 差兩個數量級。

:::

**附帶發現：排序準則本身亦為產業偏誤的來源。**
`HAN3-REV` 的跨產業配對比例 19.7%，而分群設定完全相同、僅排序準則不同的
`AGG-SSD-NF` 僅 6.8%。SSD 距離偏好同產業配對——同業股票的歷史價格路徑天然更接近。
此點在排序固定為 SSD 的因子設計中無法觀察。

> **綜合**：形成期實作差異與交易機制差異皆已檢驗，**皆非命題 1 失敗的原因**。
> 剩餘差距歸因於兩項**資料可得性限制**而非設計選擇——分群特徵 7 維連續
> （原文 48 動量因子 + 78 公司特徵）、母體 S&P 500（原文 CRSP 全市場）。

## 特徵維度與插補管道

形成期實作差異與交易機制皆已排除後，剩餘殘差之一為**特徵維度**——本研究原為
7 維連續特徵，Han et al. 為 48 動量因子 + 78 公司特徵。自 SEC companyfacts
解出 40 個 Green et al. 特徵，**12 個通過 70% 覆蓋率**（以 PIT 成分股身分
為分母），連續維度 7 → 19。

| 步驟（2012+） | 年化Δ% | BH 校正 $p$ |
| :--- | ---: | ---: |
| one-hot → 0 | −0.171 | 0.619 ✘ |
| 產業插補 → 全域插補 | +0.481 | 0.619 ✘ |
| **7 維 → 19 維連續** | −0.482 | 0.619 ✘ |

兩大效果幾乎抵消（+0.481 / −0.482），淨值約 −0.001pp。對 GICS 為 +0.283pp
（$p$=0.535），**未優於** 7 維版的 +0.455pp。**這些特徵替代了產業先驗，但未超越它。**

⚠️ 這兩個中間步驟的**符號不穩定**：補齊原始價快取前為 −0.716／+0.685，
補齊後皆反轉，而淨效果與「全部不顯著」不變——**不顯著的點估計不宜作方向性解讀**。

::: {.callout-important}

### 方法論發現：產業中位數插補是第三條產業資訊管道

`impute_by_group` 以**產業中位數**填補缺失。結構性特徵在 2012+ 有 30–50% 缺失，
故那些插補值等同為缺資料的股票加上產業標籤——與 `sector_onehot_weight` 同一種混淆，
但此前無人計入。**固定特徵、只改插補方式**，2012+ 跨產業配對比例：

| 臂 | 產業插補 | 全域插補 | 差 |
| :--- | ---: | ---: | ---: |
| AGG-BASE | 9.5% | 16.5% | +7.0 |
| **AGG-STRUCT** | **15.7%** | **59.3%** | **+42.8** |

影響在 STRUCT 臂遠大於 BASE 臂（缺失所在），機制明確。

**已封存的 F09 結構性特徵消融因此是在「處理幾乎未施加」的條件下做的**——原設定下
加 10 個特徵僅使跨產業 +6.2pp，全域插補下為 +42.8pp。

:::

**F09 重驗（3 分群 × BASE/STRUCT，唯一改動為插補方式）**：
BH 校正後 **0/6 顯著**。**原結論不但成立，且由「弱 null」（處理未真正施加）
升級為「強 null」**（處理完整施加、分群大幅改變，仍無效果）。

::: {.callout-warning}

### 這裡可以說什麼、不可以說什麼

跨產業配對比例在六種設定下自 **0%（GICS）到 69%（不分組）**，涵蓋 one-hot、插補、
特徵集、分組方式四個維度，而 2012+ 績效全部落在 **−1.84% ~ −0.66%**。

**可以陳述**：在該範圍內，本研究**未觀察到本實驗解析度所能偵測的績效差異**。

**不可陳述**：「選哪些配對不決定報酬」。六種設定的績效全距為 **1.18pp**，
而單一組對照的信賴區間寬度即為 **1.18 ~ 1.91pp**——
**全部六種設定的離散度，尚不及一組對照的區間寬度**。
本研究未施加等價檢定，此類主張須有實質等價邊界的證據，本表不提供該證據。

:::

## 形成期的比較為何難以有結論

::: {.callout-important}

### 一組自然的受控對照

門檻分位 90 的 ML 分群與 GICS 產業分組**期均配對數同為 15.0**——
候選池大小完全相同，唯一差異是跨產業比例（31.9% vs 0.0%）。

該組對照下 GICS 的點估計較高（Δ年化 −0.148pp、ΔSharpe −0.036，ML 勝 4/15 格），
惟**未達統計顯著**（$p$ = 0.140）。

:::

一項**結構性**的觀察：產業分組同時取得兩項在 ML 分群下互斥的性質。

| | 候選池大小 | 產業一致性 |
| :--- | :---: | :---: |
| ML 分群（細） | ✘ 不足 | ✔ 高 |
| ML 分群（粗） | ✔ 充足 | ✘ 被破壞 |
| **GICS** | **✔ 充足** | **✔ 完全一致（定義上）** |

ML 分群依特徵空間的幾何鄰近性切割，擴大群體必然混入其他產業的股票；
GICS 則以外生的產業定義直接給出「大且純」的分組。
候選池大小與跨產業比例的 Spearman $\rho$ = **+1.000**——**兩者在 ML 分群下無法解耦**。

::: {.callout-warning}

### 但這**不是**「命題 1 未獲支持的根本原因」

上述機制解釋的是**形成期結構**為何難以同時最佳化，而非績效差異的成因——
因為**績效差異本身就沒有被測到**。真正的障礙在檢定力，且可以量化：

| | $SE$ 中位 (pp) | MDE 中位 (pp) | 對 0.3pp 真實效果的檢定力 |
| :--- | ---: | ---: | ---: |
| 命題 1（兩臂持**不同**配對） | 0.406 | **1.14** | **11%** |
| 命題 2（兩臂**共用**配對） | 0.214 | 0.60 | — |

命題 1 的兩臂持有不同的配對集合，差分中混入兩套標的的特異變異；
命題 2 的兩臂共用配對，差分即消噪。**標準誤相差 1.89 倍，
等效於形成期層需 3.6 倍的樣本期間。**

若要求最小可偵測效果降至 0.3pp（能看見經濟上實質的差異），
形成期層約需 **14 倍**樣本、**逾三個世紀**的日資料。

:::

> **誠實界定**：資料回補後方向已轉為 **ML 優 5 組／GICS 優 4 組**，且 9 組
> BH 校正後全部不顯著。故本節結論為「**形成期層的改良在本設計下無法被驗證**」，
> 既非「GICS 顯著較優」，亦非「ML 分群無效」。
> 唯一的出路是構造**配對設計**——使兩臂持有同一批標的而僅變動分組——本研究未能做到。

# 命題 2：門檻選擇 vs. 固定門檻

**設計**：**固定形成期配對**，僅將交易端由固定門檻 Z-Score（$z$=2.0）換成
**DL-THR**（逐期由模型自 9 個動作中選擇門檻）。
配對底涵蓋三種 ML 分群與**傳統 GICS 分組**，以檢驗增益是否依賴特定配對來源。

同參數網格逐格配對檢定（$n = 15$）：

| 配對底 | ΔSharpe | 勝格數 | $p$ | 最佳年化（Z → DL-THR） |
| :--- | :---: | :---: | :---: | :---: |
| **GICS × SSD（傳統）** | **+0.402** | 13/15 | 0.0013 | 1.66% → 2.13% |
| HDBSCAN × SDP | +0.317 | 13/15 | 0.0037 | 1.70% → 2.50% |
| GICS × SDP（傳統） | +0.281 | 12/15 | 0.0070 | 1.65% → 1.97% |
| K-means × SSD | +0.285 | 14/15 | 0.0017 | 1.31% → 1.90% |
| Agglomerative × SSD | +0.263 | **15/15** | 0.0006 | 1.77% → 2.37% |

**五種配對底全部顯著改善**（$p < 0.01$）。
增益最大者為傳統 GICS 分組——**DL-THR 的價值不依賴機器學習分群**。

**穩健性**（15 格中 Sharpe 為正者）：

| 配對底 | Z-Score | → DL-THR |
| :--- | :---: | :---: |
| GICS × SSD | 5/15 | **13/15** |
| HDBSCAN × SDP | 4/15 | 8/15 |
| Agglomerative × SSD | 3/15 | 8/15 |
| K-means × SSD | 2/15 | 5/15 |


## 命題 2 的四個發現

**1. 增益普適於所有配對底，包含傳統分組**
五種配對底（3 種 ML 分群 + 2 種 GICS 排序組合）疊加 DL-THR 後全部顯著改善，
逐日 bootstrap $p$ 值介於 0.0000 ~ 0.0060。**傳統 GICS × SSD 底的增益最大
（年化 +1.105 pp，$p$ = 0.0010，且逐格顯著數最高 8/15）**，顯示 DL-THR 交易端是一個與配對來源正交的獨立改良——
此普適性比僅在 ML 配對底上驗證更強。

**2. DL-THR 大幅提升參數穩健性**
GICS × SSD 底的正 Sharpe 網格數自 5/15 增至 13/15。
DL-THR 為每組配對每期自選門檻，等同於將原本需人工調校的參數交由模型依情境決定，
因而降低對外生參數設定的敏感度。

**3. 增益的來源：三項行為面解釋皆已排除（見下節）**
DL-THR 的動作選單含 SKIP 與 8 組門檻，但受控對照顯示增益**既非**拉高進場門檻、
**亦非**篩掉劣質配對、**更非**減少曝險——DL-THR 的進場次數反而是 Z-Score 的 **1.55–1.92 倍**。
（惟門檻管道並非毫無貢獻：HDBSCAN 底複製 28.4%、$p$=0.014；
但 DL-THR 在同門檻下仍顯著勝出。）
殘差與「槽位週轉」一致，惟本研究無法直接驗證，列為後續研究。

**4. 增益也不來自「全資訊」這個運氣**
本研究另建 **RL-THR**：同一動作選單、同一 12 維狀態、同一網路與 walk-forward 切分，
僅把訓練標籤縮成「實際選中的那一個」並改採 $\varepsilon$-greedy，使其成為真正的
部分回饋（contextual bandit）。等權口徑下兩臂**無法區分**——
$\varepsilon$=0.10 時 **+0.015 pp、95% CI [−0.38, +0.45]、$p$=0.94**，
三組 $\varepsilon$ 一致（−0.12 ~ +0.02 pp）。區間兩端皆遠小於總增益 +0.787 pp，
故為「相當」而非檢定力不足。

> 配合已證偽的逐日定位動作空間（三代真強化學習，中位 Sharpe −1.1 ~ −2.3），
> 失敗與成功兩個方向指向同一結論：**關鍵是動作空間設計，不是訓練方法。**

> **DL-THR 未固定隨機種子**，上表為單次訓練值。三種 ML 配對底的五輪重訓結果見下節。

## 反事實標籤值多少錢：DL-THR vs RL-THR

配對底 `Grid (AGG-SSD)`，15 格等權、逐日、循環 block bootstrap（$n$=6,287）。

| 對照 | 年化Δ | 95% CI (pp) | $p$ |
| :--- | ---: | :---: | ---: |
| **DL-THR − Z-Score**（參照） | **+0.787 pp** | [+0.34, +1.27] | **0.0011** |
| RL-THR ($\varepsilon$=0.05) − Z-Score | +0.707 pp | [+0.10, +1.45] | **0.0400** |
| **RL-THR ($\varepsilon$=0.10) − Z-Score** | **+0.802 pp** | [+0.20, +1.54] | **0.0191** |
| RL-THR ($\varepsilon$=0.20→0.02) − Z-Score | +0.669 pp | [+0.08, +1.38] | **0.0450** |
| RL-THR ($\varepsilon$=0.05) − DL-THR | −0.081 pp | [−0.47, +0.34] | 0.6935 |
| **RL-THR ($\varepsilon$=0.10) − DL-THR** | **+0.015 pp** | **[−0.38, +0.45]** | **0.9409** |
| RL-THR ($\varepsilon$=0.20→0.02) − DL-THR | −0.118 pp | [−0.49, +0.28] | 0.5357 |

最有利排程下兩臂相差 **+0.015 pp**，區間 **[−0.38, +0.45]** 幾乎對稱橫跨零；
三組 $\varepsilon$ 一致（−0.12 ~ +0.02 pp）。兩端皆遠小於總增益 +0.787 pp
→ 這是「兩端皆小」的相當，不是檢定力不足。

::: {.callout-warning}

### 兩個報告口徑給出相反的答案

改報**最佳格**（五輪獨立重跑）：DL-THR 中位年化 **2.387%**［2.341, 2.649］
vs RL-THR $\varepsilon$=0.10 的 **2.107%**［1.985, 2.192］——
**五輪全勝且全距完全不重疊**。

不矛盾：best-of-15 放大微小而系統性的優勢（資訊量九倍的一臂更**可靠地**產出好的
極大值），等權組合把這個選擇效應平均掉。本研究一律以等權為口徑，故結論取「相當」。

**這是報告口徑重要性的具體實例：同一份資料在最佳格口徑下會支持相反的結論。**

:::

**限制**　差距同時含「資訊量僅 1/9」與「探索成本」，本設計無法分離——不探索就沒有樣本。

> 註：2026-08-10 重跑前 RL-THR 對 Z-Score 有兩組 $\varepsilon$ 僅為邊緣顯著
> （0.058 / 0.068），當時另列「bandit 自身顯著性較脆弱」為限制；
> 重跑後三組全部達 5% 顯著，該限制已不成立。

腳本：`python -m analysis.prop2_label_information`

## 命題 2 的重訓穩健性（五輪獨立訓練）

DL-THR 網路未固定隨機種子，每輪重新訓練。下表為五輪各自取網格最佳後的跨輪統計。

| 配對底 | 最佳年化 中位［範圍］ | 最佳 Sharpe 中位［範圍］ | 正 Sharpe 最少 |
| :--- | :---: | :---: | :---: |
| Agglomerative × SSD | 2.39%［2.34, 2.65］ | 0.34［0.34, 0.38］ | 6/15 |
| HDBSCAN × SSD-DTW-PCA | **2.46%**［2.41, 2.54］ | 0.31［0.31, 0.32］ | **8/15** |
| K-means × SSD | 1.73%［1.53, 1.90］ | 0.26［0.23, 0.28］ | 5/15 |

::: {.callout-important}

### 為何必須報告變異數

Agglomerative 底的單次訓練值為 **2.65%**，恰為其五輪範圍的**上界**；
其中位數 2.39% 低於 HDBSCAN 底的 2.46%。
若僅報告單次結果，將得出「Agglomerative 底的 DL-THR 表現最佳」之結論，
而該結論**無法在重訓下複現**。

跨策略比較一律以中位數為準；範圍寬度本身亦為一項結果指標
（反映該配對底提供的學習訊號穩定性）。

:::


## DL-THR 學到了什麼：決策行為解析

動作選擇未落庫，但可由 `trade_logs` 完整還原
（SKIP → 全期 `HOLD_CASH (SKIP)`；entry_z → 進場列的 $\min|Z|$）。
下表取每策略 TOP N 最大之網格（樣本最充足）。

**決策分布**（動作選單 $entry_z \in \{1.5, 2.0, 2.5, 3.0\}$，靜態基準 2.0）：

| 配對底 | 配對期數 | SKIP 率 | 門檻中位 | 1.5 / 2.0 / 2.5 / 3.0 | 偏離基準 |
| :--- | :---: | :---: | :---: | :---: | :---: |
| Agglomerative | 1462 | 35.0% | 2.22 | 12 / 40 / 21 / 27 % | **61%** |
| HDBSCAN | 1475 | 37.1% | 2.27 | 8 / 40 / 22 / 29 % | 62% |
| K-means | 1377 | 38.7% | 2.20 | 15 / 41 / 19 / 26 % | 62% |

**增益來源分解**（DL-THR − Z-Score 總損益差，逐配對拆解）：

| 配對底 | 總增益 | SKIP 貢獻 | 門檻貢獻 |
| :--- | :---: | :---: | :---: |
| Agglomerative | 3106 | 1117（36%） | **1989（64%）** |
| HDBSCAN | 845 | 260（31%） | **586（69%）** |
| K-means | 1361 | 711（52%） | 650（48%） |

::: {.callout-warning}

### SKIP 並非選擇性技巧

被 DL-THR 跳過的配對，其在 Z-Score 端的虧損比例與留下者**幾乎相同**
（Agglomerative 60% vs 60%；HDBSCAN 48% vs 51%；K-means 50% vs 48%），
Mann-Whitney 檢定 $p$ = 0.35 ~ 0.93，無一顯著。

「被跳過者損益為負」不足以證明技巧——配對平均損益本就為負，
**隨機棄權同樣會「避開虧損」**。判準須為其損益是否顯著低於留下者。

:::

**結論**：DL-THR 的價值在於**為每組配對挑選適當的進出場門檻**（62% 的決策偏離靜態基準），
而非拒絕交易。此結果在獨立資料上支持 Kim & Kim (2019) 的門檻最適化主張，
並量化了該機制的貢獻比重。

**可解釋性的邊界**：門檻選擇與排序名次的 Spearman $\rho$ 僅 −0.03 ~ +0.07，
模型並非依循「名次差則提高門檻」之類的簡單規則，
其決策為 12 維形成期特徵的非線性組合，本研究未能進一步歸因。


## 命題 2 的假設檢定（一）：相對主張

**$H_0$**：DL-THR 交易端與 Z-Score 交易端的績效相同。

檢定採**配對設計**——兩者跑在完全相同的配對、期間與參數格上，唯一差異為交易端。
市場崩盤、配對失效、成本衝擊等共同風險在相減後消去，
留下的差異序列僅含「交易端決策」一項變異。此即命題 2 檢定力遠高於絕對檢定的原因。
（附帶效果：無風險利率於兩臂相減時對消，本檢定不受 Sharpe 口徑未扣 rf 的影響。）

**主檢定**：逐日報酬差 + 循環 block bootstrap（$L$=126，10,000 次），6,287 交易日。

| 配對底 | 年化Δ (pp) | 95% CI (pp) | 資訊比率 | $p$ |
| :--- | :---: | :---: | :---: | :---: |
| GICS × SSD（傳統） | **+1.105** | [+0.58, +1.76] | 0.741 | **0.0010** |
| HDBSCAN × SDP | +0.865 | [+0.48, +1.26] | 0.575 | **0.0000** |
| Agglomerative × SSD | +0.787 | [+0.34, +1.27] | 0.583 | **0.0011** |
| GICS × SDP（傳統） | +0.659 | [+0.29, +1.07] | 0.507 | **0.0009** |
| K-means × SSD | +0.606 | [+0.20, +1.04] | 0.449 | **0.0060** |

**五種配對底全部顯著，且五個信賴區間完全落在零的右側**——
最保守的下界仍為 **+0.20 pp**，故方向性主張在最不利的區間端點下依然成立。

::: {.aside}
全程並行計算 Newey-West HAC 作為參數法對照，**14 組對照兩法結論完全一致**（命題 2 五組皆顯著、命題 1 九組皆不顯著）；完整雙欄見 `results/analysis/` 各 CSV。
:::

**逐輪複核**（三種 ML 配對底各五輪獨立重訓，以舊網格配對口徑複核方向一致性）：
Agglomerative 15/15、14/15、15/15、15/15、15/15；HDBSCAN 10~13/15；K-means 11~14/15。
五輪 Sharpe 全距僅 0.013 ~ 0.047，遠小於效果量，增益非特定訓練批次所致。

> 逐格檢定顯示：ML 配對底 15 格中僅 5–6 格能**單獨**達顯著（GICS 底 7–8 格）。
> 惟此處的顯著**不是檢定力僥倖**：點估計中位 +0.787pp 明顯大於該比較自身的
> 最小可偵測效果（0.60pp）。
> 等權組合顯著係因平均掉各格特異噪音；**單一參數設定的檢定力不足**，此點須據實揭露。

## 命題 2 的假設檢定（二）：絕對主張

**H0**：策略的平均日報酬為零。此處無對照組，須直接對抗市場噪音。
口徑同全篇：**15 格等權組合** → 沒有東西可挑，故無選擇偏誤需要校正。

| 配對底 | 交易端 | 年化 | 95% CI (pp) | $p$ |
| :--- | :--- | ---: | :---: | ---: |
| Agglomerative | Z-Score | −0.03% | [−1.07, +1.24] | 0.965 |
| Agglomerative | DL-THR | +0.68% | [−0.48, +2.12] | 0.303 |
| HDBSCAN | Z-Score | −0.05% | [−1.25, +1.48] | 0.945 |
| HDBSCAN | DL-THR | +0.77% | [−0.50, +2.41] | 0.299 |
| K-means | Z-Score | −0.36% | [−1.35, +0.77] | 0.516 |
| K-means | DL-THR | +0.27% | [−0.83, +1.53] | 0.659 |

**六個區間全部橫跨零，無一顯著。** 三個 Z-Score 臂點估計皆負、三個 DL-THR 臂皆正，
方向與上一節一致。

::: {.callout-warning}

### 本表與上一節的數字**不可直接相減**

本表的日報酬採**複利**口徑（`Daily_Delta` / 前一日權益），與引擎依當期權益配置
部位的實際行為一致；上一節的差分檢定則採**單利**口徑（日損益金額 / 期初資金）。

兩個複利序列的差，不等於同一組資料在單利口徑下的差分——
Agglomerative 本表相減得 **+0.71pp**，上一節為 **+0.79pp**。
方向與量級一致，差距源於口徑而非資料，**故不作為自洽性檢核使用**。

:::

::: {.callout-important}

### 這是結構性限制，不是樣本不足

等權組合的年化 Sharpe 僅 **0.09 ~ 0.21**，而絕對檢定的 $t \approx SR\sqrt{\text{年數}}$
（樣本 6,287 交易日 ≈ 24.9 年）：

| 配對底（DL-THR 臂） | 年化 Sharpe | 實測 $t$ | 達 $p<0.05$ 所需年數 |
| :--- | ---: | ---: | ---: |
| HDBSCAN | 0.209 | 1.04 | **88** |
| Agglomerative | 0.206 | 1.03 | 91 |
| K-means | 0.088 | 0.44 | 493 |

**本研究不主張任何單一策略的絕對獲利能力具統計顯著性。**

:::

> **附錄的 Deflated Sharpe**：若改報「網格最佳格」，數字較高但內含 15 選 1 的偏誤。
> 以實地清點的試驗宇宙 $N$ = 110 計，共同門檻 $SR_0$ = **0.408**（年化），
> 而上表六族中最高的 $SR$ 僅 **0.392**——DSR 全數落在 **0.256 ~ 0.467**，
> 無一通過 0.95。
>
> $N$ = 110 含為隔離前行研究差異而新增的 `HSU25 (…-REV)` 三條（結論為負，
> 但仍各佔一次試驗——**試驗數不因結果不利而豁免**）。
> 較廣的策略族全表見 `breakeven_dsr.csv`（同一次清點，$N$ = 110）：
> **98 個策略族亦無一通過 0.95**，最高者 Grid (AGG-SSD-NF-NOSEC) 的
> $SR$ = 0.471、$DSR$ = 0.632。注意論證方向：該表門檻 $SR_0$ = 0.408
> **低於**其最高 $SR$，故結論由 DSR 檢定本身承擔。
>
> **此表不構成正文的主張。**

## 兩項檢定為何結論相反

::: {.callout-important}

### 相對顯著、絕對不顯著，並不矛盾

兩者的**虛無假設不同**：

| | 逐日差分 bootstrap | 絕對 bootstrap / DSR |
| :--- | :--- | :--- |
| $H_0$ | 兩交易端績效相同 | 策略平均日報酬（或真實 Sharpe）為零 |
| 對照物 | 同配對、同參數格的 Z-Score | 零 |
| 主張性質 | 相對 | 絕對 |
| 校正對象 | 自相關（block bootstrap / HAC） | 多重測試偏誤（$N$ = 110） |

配對設計消去了兩策略共同承受的市場風險，訊噪比大幅提高；
絕對檢定沒有這個對照，必須從市場噪音中直接辨識訊號。

:::

**檢定力受限於低 Sharpe，而非樣本長度。**
等權組合的年化 Sharpe 僅 0.09 ~ 0.21，理論 $t \approx SR\sqrt{24.9}$ 與實測值一致
（HDBSCAN 臂 1.04 vs 1.04）。以最佳臂計，欲達 $p<0.05$ 約需 **88 年**樣本。
此為成本後配對交易的普遍特性——Do & Faff (2010) 記錄了配對交易報酬自 2000 年代
起的長期衰減。

**形成期的比較天生比交易期難——且難在設計，不在資料量。**
命題 1 的兩臂為**不同的配對集合**（特異變異無從消去），
命題 2 的兩臂**共用同一批配對**（差分即消噪）。
兩者標準誤相差 **1.89 倍**，等效於形成期層需 **3.6 倍**樣本；
對 0.3pp 的真實效果，命題 1 的檢定力僅 **11%**。
這解釋了為何同一套方法在命題 2 上得到 $p$ < 0.01、在命題 1 上卻全部不顯著。

> **本研究的檢定定位**：命題 2 與組合系統為**方法之相對優劣**的假設檢定，證據充分；
> 絕對獲利能力則未達統計顯著，此點列於限制章。

# 組合系統：實務上要部署的那個檢定

命題 1 與命題 2 各測**一個成分**，**都不是實務上要部署的系統**。

> **組合系統**　動態分群（252 日窗、21 日滾動、295 期）+ SSD/SDP 排序
> + 共整合篩選 + **DL-THR 門檻選擇**
>
> **傳統基準**　GICS 產業分組 + 同一排序 + 同一篩選 + **固定門檻 Z-Score**

| 期間 | 分群法 | 傳統基準 | 年化Δ(pp) | IR | 95% CI (pp) | BH 校正 $p$ | |
| :--- | :--- | :--- | ---: | ---: | :---: | ---: | :---: |
| 全期 | Agglomerative | GICS-SSD | **+1.156** | 0.455 | [+0.32, +2.01] | **0.0141** | **✔** |
| 全期 | HDBSCAN | GICS-SDP | **+1.025** | 0.412 | [+0.31, +1.81] | **0.0141** | **✔** |
| 全期 | K-means | GICS-SSD | +0.631 | 0.239 | [−0.31, +1.56] | 0.1864 | ✘ |
| 2012+ | Agglomerative | GICS-SSD | **+1.511** | 0.762 | [+0.53, +2.57] | **0.0074** | **✔** |
| 2012+ | HDBSCAN | GICS-SDP | **+1.194** | 0.647 | [+0.36, +2.06] | **0.0074** | **✔** |
| 2012+ | K-means | GICS-SSD | **+1.193** | 0.576 | [+0.17, +2.21] | **0.0203** | **✔** |

**六組中五組於 BH 校正後達 5% 顯著**（排序準則已對齊，否則會混入分組與排序兩個變因）。

## 成分分解：改善來自哪一半？

$$(\text{分群}+\text{DL-THR}) - (\text{GICS}+\text{ZS}) = \underbrace{\cdots}_{\text{DL-THR 成分}} + \underbrace{\cdots}_{\text{分群成分}}$$

| 期間 | 分群法 | 總效果 | DL-THR 成分 | ($p$) | 分群成分 | ($p$) |
| :--- | :--- | ---: | ---: | ---: | ---: | ---: |
| 全期 | Agglomerative | +1.156 | **+0.787** | **0.0011** | +0.369 | 0.363 |
| 全期 | HDBSCAN | +1.025 | **+0.865** | **0.0000** | +0.160 | 0.598 |
| 全期 | K-means | +0.631 | **+0.606** | **0.0060** | +0.025 | 0.954 |
| 2012+ | Agglomerative | +1.511 | **+0.743** | **0.0008** | +0.768 | 0.122 |
| 2012+ | HDBSCAN | +1.194 | **+1.000** | **0.0000** | +0.195 | 0.612 |
| 2012+ | K-means | +1.193 | **+0.464** | **0.0200** | +0.730 | 0.140 |

分解殘差 $\le$ 0.001 pp。

::: {.callout-important}

### DL-THR 成分六組全部顯著；分群成分六組全部不顯著

1. **分群成分不再是負的。** 重跑前三組中有兩組為負（−0.182、−0.106），
   舊版據此宣稱「分群成分**侵蝕**了改善」——資料回補後六組**全部轉正**，
   該宣稱不再成立、不應再被引用。
2. **但方向為正 $\ne$ 有效。** 六組分群成分的 $p$ 值介於 0.122 ~ 0.954，無一顯著。

→ **完整系統顯著優於傳統基準；但把功勞歸給哪一半，資料還答不出來。**

:::

::: {.callout-warning}

### 這是相對主張

同一批系統的**絕對**績效在等權口徑下六組全部不顯著（見前節），
兩者虛無假設不同，故不矛盾——但兩句必須同時陳述。

此外，此結論在 2026-08-10 的重跑後才成立：重跑前六組**無一顯著**
（BH 校正 $p$ = 0.071 ~ 0.312）。

:::

## 命題 2 的 Regime 穩健性

以等權市場的 63 日滾動波動率三分位與 126 日趨勢標記每個交易日，
分層計算年化 Sharpe。檢驗增益是否僅來自特定市場環境。

口徑同全篇：**15 格等權、逐格對齊兩臂** → 唯一變因仍是交易端。

| 配對底 / 交易端 | Calm | Normal | Turbulent | Bull | Bear |
| :--- | :---: | :---: | :---: | :---: | :---: |
| Agglomerative × SSD | −0.81 | −0.60 | 0.79 | −0.23 | 0.41 |
| Agglomerative **+ DL-THR** | **−0.38** | **−0.36** | **0.95** | **+0.08** | **0.59** |
| HDBSCAN × SDP | −0.64 | −0.55 | 0.63 | −0.17 | 0.29 |
| HDBSCAN **+ DL-THR** | **−0.20** | **−0.09** | **0.66** | **+0.15** | **0.46** |
| K-means × SSD | −0.71 | −0.60 | 0.41 | −0.22 | −0.02 |
| K-means **+ DL-THR** | **−0.55** | **−0.12** | **0.48** | **−0.03** | **0.29** |
| GICS × SSD | −1.24 | −0.52 | 0.54 | −0.44 | 0.31 |
| GICS × SSD **+ DL-THR** | **−0.72** | **−0.15** | **0.70** | **−0.04** | **0.52** |
| GICS × SDP | −0.69 | −0.77 | 0.68 | −0.27 | 0.31 |
| GICS × SDP **+ DL-THR** | **−0.49** | **−0.32** | **0.75** | **−0.01** | **0.47** |

**DL-THR 在 25 個 regime 格中改善 25 格、劣化 0 格。**
命題 2 的增益不依賴特定市場環境。

⚠️ **口徑更正（2026-08-12）**：初稿以網格最佳配置計算、報告「20 格改善 19 格」。
兩臂各自取最佳格時選到的格子不同（HDB-SDP 是 Top3、HDB-SDP-DRL 是 Top1），
比較混入了「換格子」這個變因；重跑後該口徑退為 12 改／8 劣。
等權逐格對齊才是乾淨的比較。

**同時揭露一項策略層面的限制**：配對交易的獲利完全集中於
**高波動與空頭**環境；**平靜期十組配置全部為負 Sharpe**（−1.24 ~ −0.20），
DL-THR 只能減輕虧損而無法反轉。
此與本研究另一項發現一致：損益集中於高分散度期間（附錄 D 的 regime 閘門依據）。


## 成本敏感度：Break-even 分析

成本模型可解析求解——進出場費用 = friction × 名目額，
且每配對名目額恰等於每配對資金，故往返 break-even
$c^* = 2\,(0.29\% + 	ext{淨利} / \Sigma	ext{名目額})$。

| 配對底 | Z-Score | → DL-THR | 差（bps） | Z 餘裕 | DL 餘裕 |
| :--- | :---: | :---: | :---: | :---: | :---: |
| Agglomerative × SSD | 0.582% | **0.601%** | +1.9 | +0.2 | **+2.1** |
| HDBSCAN × SDP | 0.582% | **0.604%** | +2.1 | +0.2 | **+2.4** |
| K-means × SSD | 0.562% | **0.591%** | +2.9 | **−1.8** | +1.1 |
| GICS × SSD | 0.568% | **0.594%** | +2.7 | **−1.2** | +1.4 |
| GICS × SDP | 0.576% | **0.592%** | +1.6 | **−0.4** | +1.2 |

*往返 break-even 成本；現行假設 0.58%（單邊 0.29%，Do & Faff 2012）。
口徑：15 格等權、逐格對齊兩臂*

**DL-THR 在五種配對底上一致提高成本承受度**（+1.6 ~ +2.9 bps）。機制為 SKIP 與
較高的進場門檻共同減少交易次數，使同樣的毛利分攤在更少的名目額上。

::: {.callout-important}

### 但餘裕本身薄到接近於零

十組配置的餘裕介於 **−1.8 ~ +2.4 bps**，且 **K-means／GICS-SSD／GICS-SDP
三個 Z-Score 臂已經為負**——現行假設下它們本來就不獲利。
DL-THR 把五個配對底全部推回正值，幅度僅 1~2 bps。

⚠️ 初稿報告的 6.5–12.8 bps 取自**網格最佳格**，內含 15 選 1 的選擇偏誤。
等權口徑才與絕對績效一致：等權年化本就落在 −0.36% ~ +0.77%、無一顯著，
其 break-even 理當貼著成本假設本身。

**本研究之結論應限於方法間的相對比較。**

:::


# 附錄摘要

主軸之外的支撐性實驗，完整數據見 `results/result.db` 與
`archive/config_archived_strategies.py`。

| 附錄 | 內容 | 主要結果 |
| :--- | :--- | :--- |
| **A** | 文獻原始設定復現 | Gatev (2006) 原型、許鈞翔 (2025) ADF 0.01 設定；Grid (GICS-SSD) 與原生 SSD Rolling 數值完全相同 |
| **A′** | 前行研究差異的受控定位 | 與許鈞翔 (2025) 的八項差異清單；隔離「進場時點」一項：回歸式進場使三種排序年化全部下降 −0.77 ~ −0.91pp，BH 後 0/3 顯著（`analysis/hsu25_entry_timing.py`） |
| **B** | 篩選消融（有/無三道統計篩選） | 篩選貢獻 +0.25 ~ +0.87pp，三種排序下皆為正 |
| **C** | 特徵層消融 | 多尺度動量（−0.94 ~ −2.43pp）、SEC 結構性財報比率（±0.22pp 噪音範圍）皆無助益 |
| **D** | 延伸探索：regime 條件化進場 | 低分散度閘門使全網格 Sharpe 轉正、MDD 下降；三層疊加五輪中位 2.69%［2.65, 2.71］、最差輪 14/15 正 Sharpe |
| **I** | Regime／成本可重現腳本 | `analysis/regime_cost_dsr_eval.py`：regime 分層 Sharpe、break-even 成本、DSR |
| **H** | 粒度掃描可重現腳本 | `analysis/granularity_sweep.py`：切割門檻 × 候選池 × 跨產業比例 × 績效 |
| **G** | 行為解析可重現腳本 | `analysis/drl_behavior.py`：決策分布／SKIP 技巧性／增益來源分解 |
| **F** | 統計檢定可重現腳本 | `analysis/proposition2_stats.py`：配對 t／Wilcoxon／逐輪檢定／Newey-West／DSR 四項一次產出 |
| **E** | 參數敏感性 | ADF 門檻 0.01 / 0.05 / 0.1 對照，說明本研究採 0.05 的實證依據 |
| **J** | 反事實標籤的價值 | `analysis/prop2_label_information.py`：RL-THR（部分回饋 + ε-greedy）vs DL-THR（全資訊監督），等權口徑下相當 |

## 附錄 B、E 的方法論意涵（值得在正文引用）

**篩選是必要的**：純距離排序不足以識別可交易配對——距離度量回答「歷史走勢多接近」，
共整合檢定回答「價差是否會回歸」，兩者結合才構成有效選取（SSD 排序下 0.79% → 1.66%）。

**ADF 門檻 0.05 優於文獻慣用的 0.01**：實證顯示過嚴門檻反而降低績效
（Agglomerative FMP：1.27% @0.01 vs 1.77% @0.05），且 0.05 → 0.1 已飽和。
機制為「排序流程先按距離排序、再逐一檢定並填滿 top_n」——門檻收緊迫使系統
往距離更遠的候選尋找，**以經濟相似性換取統計顯著性，淨效果為負**。
候選池充足時不再出現早期「篩選過嚴導致無配對」的問題（ADF 0.01 仍可填滿 20/20 對）。


# 結論與限制

## 結論

0. **組合系統顯著優於傳統基準。** 將兩層重新組裝為實務上要部署的那個系統
   （動態分群 + DL-THR 門檻選擇 vs GICS + 固定門檻 Z-Score），
   六組對照中**五組於 BH-FDR 校正後達 5% 顯著**（+0.63 ~ +1.51pp）。
   成分分解顯示 **DL-THR 成分六組全部顯著、分群成分六組全部不顯著**
   （$p$ = 0.122 ~ 0.954）。
   **完整系統顯著優於傳統基準，但把功勞歸給哪一半，資料還答不出來。**
   此為相對主張——同一系統的絕對績效在等權口徑下全部不顯著。

1. **命題 1 未獲支持**：機器學習分群未能建立優於 GICS 產業分類的配對搜尋空間。
   9 組逐日對照經 BH-FDR 校正後**無一顯著**（校正後最小 $p$ = 0.455），
   方向上 ML 優 5 組、GICS 優 4 組。此結果取自 2026-08-10 基本面資料回補後的
   重跑——回補前方向為 1:8，回補後仍無一顯著，故已排除「資料太稀疏」的替代解釋。
   **資料條件改變的是方向，不是顯著性。**
   此為「未獲支持」，**非**「GICS 顯著較優」；信賴區間寬 1.2 ~ 1.9pp，
   是 GICS 參照臂自身績效（−0.11% ~ −0.40%）的數倍，**亦無法主張兩者相當**。
   受控的粒度掃描進一步顯示：**分群演算法並非關鍵變因**——
   單一切割門檻參數造成的績效變動（2.05pp）遠大於三種演算法間的差距（0.48pp）。
   **命題 1 的三項可能實作原因皆已逐一檢驗且皆非失敗原因**：形成期設計
   （產業 one-hot、共整合篩選、缺不分組零點）、交易機制（Han et al. 的四項差異
   逐步施加）、特徵維度（7 → 19 維連續）。

1b. **核心發現：形成期的改良難以驗證，且難在設計、不在資料量。**
   命題 1 的兩臂持有不同的配對集合（特異變異無從消去），命題 2 的兩臂共用配對
   （差分即消噪）。兩者標準誤相差 **1.89 倍**，等效於形成期層需 **3.6 倍**樣本；
   對 0.3pp 的真實效果，命題 1 的檢定力僅 **11%**——即使命題 1 為真，
   本實驗仍有近九成機率報告「不顯著」。**故此一 null 對命題 1 的真偽幾乎不具鑑別力。**
   輔證：跨產業配對比例在六種設定下自 **0%（GICS）到 69%（不分組）**，
   涵蓋 one-hot 權重、插補方式、特徵集、分組方式四個維度，
   2012+ 績效全部落在 **−1.84% ~ −0.66%**、未觀察到可偵測的差異；
   惟其全距（1.18pp）尚不及單一對照的區間寬度（1.18 ~ 1.91pp），
   **故此為輔證，不構成「配對選擇不影響報酬」的等價證據**。

1c. **方法論發現：產業中位數插補是先前未被計入的第三條產業資訊管道。**
   `impute_by_group` 對缺失值填產業中位數；結構性特徵在 2012+ 缺失 30–50%，
   故等同為缺資料的股票加上產業標籤。固定特徵、僅改插補方式，AGG-STRUCT 的
   跨產業配對比例即自 15.7% 升至 59.3%（+42.8pp），而 BASE 臂僅 +7.0pp。
   已封存的 F09 結構性特徵消融因此是在「處理幾乎未施加」下執行；經全域插補重驗，
   其「無貢獻」結論**仍成立**，並由弱 null 升級為強 null（3 分群 × BASE/STRUCT
   共 6 組，BH 校正後 0/6 顯著）。

2. **命題 2 獲得支持**：以學習法選擇門檻（DL-THR）在**五種配對底**上皆顯著優於
   固定門檻 Z-Score（循環 block bootstrap $p$ = 0.0000 ~ 0.0060，Newey-West HAC 一致），
   五個 95% CI 完全落在零的右側（最保守下界 +0.20pp），
   且增益最大者為傳統 GICS 分組——此改良與配對來源正交，不依賴機器學習分群。
   與命題 1 不同，此處的顯著**不是檢定力僥倖**：點估計中位 +0.787pp
   明顯大於該比較自身的最小可偵測效果（0.60pp）。

3. **增益具 regime 穩健性與成本穩健性**：以 15 格等權、逐格對齊兩臂的口徑計，
   DL-THR 在 **25 個市場環境分層中改善 25 格、劣化 0 格**，
   並在**五種配對底上一致提高**往返 break-even 成本（+1.6 ~ +2.9 bps）。
   ⚠️ 初稿的「20 格改善 19 格」取自網格最佳格，兩臂最佳格不同故混入變因；
   重跑後該口徑退為 12 改／8 劣，等權逐格對齊才是乾淨的比較。

4. **增益機制：四項替代解釋已排除**。受控對照顯示增益非源自拉高門檻
   （同門檻 2.2 對照下五支仍顯著）、非源自篩掉劣質配對（SKIP 置換檢定 75 格
   僅 6 格顯著，隨機期望 3.75 格）、非源自減少曝險（DL-THR 進場次數為
   Z-Score 的 1.55–1.92 倍）。
   ⚠ **惟門檻管道並非毫無貢獻**：HDBSCAN 底的 ZS(2.2)−ZS(2.0) 達 +0.246pp
   （$p$=0.014），複製 **28.4%** 的表面增益；五組的複製率介於 −8.9% ~ 28.4%，
   **一組顯著**。DL-THR 在同門檻下仍顯著勝出，故結論維持，但不可再說門檻管道無貢獻。
   殘差與槽位週轉一致，惟無法直接驗證。
   第四項為問題性質而非行為：把全資訊反事實標籤換成真正的部分回饋
   （RL-THR，同動作空間、同狀態、$\varepsilon$-greedy），等權口徑下兩臂**相當**
   （+0.015 pp，95% CI [−0.38, +0.45]，$p$=0.94）→ 增益不來自「這個問題碰巧是全資訊的」。
   結合已證偽的逐日定位動作空間，**關鍵是動作空間設計，不是訓練方法**。

5. **方法論觀察**：抽樣單位的選擇會改變推論結論。以「參數網格」為觀測單位
   構成 pseudo-replication（15 格共用同一條回測路徑），改以時間為單位後，
   命題 2 的效果量幾乎不變（+0.787 vs +0.732pp）而顯著性獲得正當基礎，
   命題 1 的「顯著較差」則消失。

## 限制

- **策略之絕對獲利能力未達統計顯著**：等權組合六組 bootstrap 檢定全部不顯著
  （區間寬 2.4 ~ 2.9pp）；Deflated Sharpe 在 $N$ = 110 的試驗宇宙下，
  共同門檻 $SR_0$ = 0.408 而六族最高 $SR$ 僅 0.392，DSR 落在 0.256 ~ 0.467，
  **無一通過 0.95**。（較廣的 98 個策略族全表同一次清點，最高 $DSR$ 0.632，
  亦為 **0/98**。）
  本研究之結論限於方法間的相對比較，不宣稱策略具可實現之超額報酬
- **形成期層的檢定力不足，且無法以更多資料補救**：標準誤較交易期層大 1.89 倍，
  需 3.6 倍樣本才能達到相同解析度；若要求最小可偵測效果降至 0.3pp，
  約需 14 倍樣本、逾三個世紀的日資料。唯一的出路是構造**配對設計**
  ——使兩臂持有同一批標的而僅變動分組——本研究未能設計出這樣的對照
- **公司特徵僅取得可得子集**：Han et al. 使用 48 動量因子 + 78 公司特徵；
  本研究自 SEC XBRL 建出 40 個，僅 **12 個**通過 70% 覆蓋率門檻（連續維度 7 → 19）。
  補齊原始價快取後 `bm`、`ep` 已入選，惟仍有 5 個評價類特徵
  （`cfp`／`cashpr`／`sp`／`dy`／`lev`）落在 57–70% 未達門檻——其缺口來自 XBRL
  標記本身而非價格資料，補價格已無法再改善。另有 11 個特徵**永遠無法取得**
  （員工數、Compustat 上市年資、sin 股分類、可轉債／擔保債旗標、財報公布日 EPS 意外）
- **命題 2 的單一設定檢定力不足**：等權組合顯著，但 ML 配對底 15 格中僅
  5–6 格能單獨達顯著（GICS 底 7–8 格）；面對更保守的固定門檻基準（$entry_z$=2.5）時，
  五支中三支失去 5% 顯著（$p$ = 0.065 ~ 0.145），Agglomerative 退為邊緣（0.058）
- **與前行研究回測結果的差距未能完全解釋**：與許鈞翔（2025）逐項比對後辨識出
  **八項**實作差異，本研究僅隔離其中一項（**進場時點**）施加受控檢定。
  結果為改採其回歸式進場後三種排序年化**全部下降**（−0.77 ~ −0.91pp，
  BH 校正後 **0/3** 顯著）——方向與待解釋的差距**相反**，故此項不是差距的來源，
  但**差距本身仍未獲解釋**。其餘七項尤以成本假設（0.29% 對 0.10%）與報酬口徑
  （$R^{CC}$ 對 $R^{EC}$）未予檢驗。該三條策略已計入 DSR 試驗宇宙
- **存活者偏誤未完全消除**：170 檔已下市成分股中 49 檔（29%）無價格資料，
  且缺失與下市狀態相關（無價格者 43% 已下市 vs 有價格者 17%）。
  方向上使絕對績效被高估（故絕對結論偏保守）、對相對比較則同向抵銷
- DL-THR 未固定隨機種子（已改以五輪中位數±範圍報告）；
  五輪對估計中位數尚屬有限，變異數本身的信賴區間未予量化
- DL-THR 的門檻選擇無法歸因至可解釋的簡單規則，模型內部決策依據仍為黑箱
- **RL-THR 對照的限制**：兩臂差距同時含「資訊量僅 1/9」與「探索成本」，
  本設計無法分離——不探索就沒有樣本。對照僅在單一配對底、單一訓練輪的
  逐日序列上執行。（2026-08-10 重跑後 RL-THR 對 Z-Score 三組 $\varepsilon$
  全部達 5% 顯著，早期「bandit 自身顯著性較脆弱」的限制已不成立）
- **成本餘裕極薄**：等權組合往返 break-even 介於 0.562% ~ 0.604%，較現行假設 0.58%
  僅相差 **−1.8 ~ +2.4 bps**，且三個 Z-Score 臂**已為負**；且該假設取自
  Do & Faff (2012) 對 1962–2009 的估計，套用於 2000–2025。
  （早期的 6.5–12.8 bps 取自網格最佳格，內含 15 選 1 偏誤，已更正）
- **獲利集中於特定市場環境**：平靜期十組配置**全部**為負 Sharpe（−1.24 ~ −0.20），
  獲利完全來自高波動與空頭環境；DL-THR 可減輕平靜期虧損但無法反轉
- 樣本限於 S&P 500 大型股，結論未必外推至中小型股或其他市場